In [1]:
# import
from pathlib import Path
import re

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap  # umap-learn

# paths
DATA_ROOT   = Path.home() / "vambersky_t/Data"
EMBED_ROOT  = DATA_ROOT / "embeddings"

# configuration
N_BINS      = 200      # 50 bp bins across 10 kb window
EMBED_DIM   = 4096     # blocks.28.mlp.l3 hidden size

RANDOM_SEED = 42
N_PCA_COMPONENTS = 50   # fit PCA on full 4096-d, keep 50 for downstream use
N_UMAP_COMPONENTS = 2

# style
sns.set_theme(style="ticks", context="paper", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


/home/jovyan/envs/evo2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_filename(npz_path: Path) -> dict:
    """
    Extract accession, TF, and biosample from filename.
    Expected format: ACCESSION__TF__BIOSAMPLE.embeddings.npz
    """
    stem = npz_path.name.split(".embeddings")[0]
    accession, tf, biosample = stem.split("__")
    return {"accession": accession, "tf": tf, "biosample": biosample}

In [3]:
npz_files = sorted(EMBED_ROOT.glob("*/*.embeddings.npz"))

if not npz_files:
    raise FileNotFoundError(f"No embedding files found under {EMBED_ROOT}. "
                            "Run 02_embeddings.ipynb first.")

print(f"Found {len(npz_files)} embedding file(s):")
for f in npz_files:
    print(f"  {f.parent.name}/{f.name}")



Found 2 embedding file(s):
  MYC/ENCFF700CXD__MYC__H1.embeddings.npz
  MYC/ENCFF765CKK__MYC__GM12878.embeddings.npz


In [4]:
# Cell 3 — Load embeddings and assemble metadata

embeddings_list = []   # list of (n_peaks, 200, 4096) float16 arrays
meta_rows = []         # one dict per file, expanded to per-peak rows below

for npz_path in npz_files:
    info = parse_filename(npz_path)
    parquet_path = npz_path.with_suffix("").with_suffix("").with_name(
        npz_path.name.split(".embeddings")[0] + ".peak_ids.parquet"
    )

    if not parquet_path.exists():
        print(f"  WARNING: sidecar missing for {npz_path.name}, skipping")
        continue

    data = np.load(npz_path)
    emb  = data["embeddings"]   # (n_peaks, 200, 4096) float16
    ids  = pl.read_parquet(parquet_path)

    assert emb.shape[0] == len(ids), (
        f"Shape mismatch in {npz_path.name}: "
        f"{emb.shape[0]} embeddings vs {len(ids)} peak IDs"
    )
    assert emb.shape[1:] == (N_BINS, EMBED_DIM), (
        f"Unexpected embedding shape in {npz_path.name}: {emb.shape}"
    )

    embeddings_list.append(emb)

    # tag each peak with file-level metadata
    n = len(ids)
    meta_rows.append(
        ids.with_columns([
            pl.lit(info["accession"]).alias("accession"),
            pl.lit(info["tf"]).alias("tf"),
            pl.lit(info["biosample"]).alias("biosample"),
            pl.lit(npz_path.parent.name + "/" + npz_path.name).alias("bed_file"),
        ])
    )

    print(f"  Loaded {n:>6} peaks  {info['tf']:<6}  {info['biosample']:<12}  "
          f"{emb.nbytes / 1e9:.2f} GB")

# ── Concatenate ───────────────────────────────────────────────────────────────
meta = pl.concat(meta_rows)
meta_pd = meta.to_pandas()

print(f"\nTotal peaks loaded: {len(meta)}")
print(f"TFs present:        {meta['tf'].unique().to_list()}")
print(f"Biosamples present: {meta['biosample'].unique().to_list()}")
print(f"Files loaded:       {meta['bed_file'].n_unique()}")

  Loaded   4957 peaks  MYC     H1            8.12 GB
  Loaded   2216 peaks  MYC     GM12878       3.63 GB

Total peaks loaded: 7173
TFs present:        ['MYC']
Biosamples present: ['GM12878', 'H1']
Files loaded:       2
